# Policy Bank: reproduce the paper's numbers

This notebook recomputes the headline results of *Policy Bank: Development-Selected Routing of Public Robot Policies for BEHAVIOR-1K* from the released data. It needs no GPU and no simulator.

[Paper](https://policy-bank.vercel.app/paper.pdf) · [Project page](https://policy-bank.vercel.app) · [Code](https://github.com/FungousLand1941/policy-bank)

In [ ]:
import os, subprocess, sys
if not os.path.exists('policy-bank'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/FungousLand1941/policy-bank'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'policy-bank[dev]'], check=True)
os.environ['POLICYBANK_DATA'] = os.path.abspath('policy-bank/data')
sys.path.insert(0, os.path.abspath('policy-bank'))

## 1. Challenge score from the 1,000 reported episodes

In [ ]:
from policybank import data as D
from policybank.metrics import bootstrap_q, challenge_q

episodes = D.load_official()
q, lo, hi = bootstrap_q(episodes, n_boot=10_000, seed=0)
print(f'Q = {challenge_q(episodes):.5f}  (95% CI {lo:.3f}-{hi:.3f})')
print('successes:', sum(e.success for e in episodes))

## 2. The frozen route and its hash

In [ ]:
from policybank.routing import Route

route = Route.from_json(D.load_route())
print(route.sha256)
for policy, tasks in route.images().items():
    print(f'{policy:22s} {len(tasks):3d} tasks')

## 3. Re-run the stage-2 selection (tasks 50-99)

In [ ]:
from policybank.analysis import reproduce_stage2

r = reproduce_stage2()
print(f"agrees on {r['agree']}/{r['tasks']} tasks using {r['dev_rollouts_used']} development rollouts")
r['disagreements']

## 4. Where the optimism sits, and the simulation study

In [ ]:
from policybank.analysis import calibration_breakdown, simulation_study

b = calibration_breakdown()
for depth, g in b['by_depth'].items():
    print(f"own dev rollouts {depth:>2}: {g['tasks']:2d} tasks, expected {g['expected']:.3f}, realized {g['realized']:.3f}")

sim = simulation_study(sims=100, curve_sims=30)   # the paper uses 300 worlds
for rule, v in sim['rules'].items():
    claimed = '-' if v['claimed'] is None else f"{v['claimed']:.3f}"
    print(f"{rule:22s} true {v['true']:.3f}  claimed {claimed}")